# BrainMapNet: Model EvaluationThis notebook evaluates trained models and generates visualizations.

In [ ]:
!pip install imageio scikit-learn opencv-python seaborn

In [ ]:
DATA_ROOT = "/content/dataset_combined_vertical"CHECKPOINT_DIR = "/content/checkpoints/fmri_cnn/"  # Change thisARCHITECTURE = "resnet18"  # or "mobilenet"MODEL_TYPE = "cnn"  # or "mlp"DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import osimport torchimport torch.nn as nnimport torchvision.models as modelsfrom torchvision.models import ResNet18_Weights, MobileNet_V3_Small_Weightsfrom torch.utils.data import Dataset, DataLoaderimport numpy as npfrom imageio.v2 import imreadfrom PIL import Imagefrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import confusion_matrix, classification_report, f1_score, roc_auc_scorefrom torch.nn.functional import softmaximport matplotlib.pyplot as pltimport seaborn as snsimport cv2print(f"PyTorch: {torch.__version__}, Device: {DEVICE}")

## Dataset Code

In [ ]:
"""EECS 442 Final Project - BrainMapNetfMRI Dataset    Loads fMRI brain activation maps from subject folders    Structure: dataset_combined_vertical/{subject}/combined_vertical/brain_cond{1-12}_vertical.png    Labels: cond1-6 = house (1), cond7-12 = face (0)"""import osfrom typing import Optionalimport torchfrom torch.utils.data import Dataset, DataLoaderimport numpy as npimport numpy.typing as nptfrom imageio.v2 import imreadfrom PIL import Imagefrom sklearn.model_selection import train_test_splitimport sysimport ossys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))from utils import config, set_random_seed# Import ImageStandardizer from train_commontry:    from train_common import ImageStandardizerexcept ImportError:    # Fallback: define ImageStandardizer here if needed    import numpy as np    import numpy.typing as npt        class ImageStandardizer:        """Standardize a batch of images to mean 0 and variance 1."""        def __init__(self) -> None:            self.image_mean = None            self.image_std = None                def fit(self, X: npt.NDArray) -> None:            """Calculate per-channel mean and standard deviation from dataset X."""            self.image_mean = np.mean(X, axis=(0, 1, 2))            self.image_std = np.std(X, axis=(0, 1, 2))                def transform(self, X: npt.NDArray) -> npt.NDArray:            """Return standardized dataset given dataset X."""            standardized = (X - self.image_mean) / (self.image_std + 1e-8)            return standardized__all__ = [    "get_train_val_test_loaders",     "get_train_val_test_datasets",     "resize",     "fMRIDataset"]def get_train_val_test_loaders(data_root: str, batch_size: int, image_dim: int = 224, **kwargs) -> tuple[DataLoader, DataLoader, DataLoader]:    """Return DataLoaders for train, val and test splits."""    tr, va, te, _ = get_train_val_test_datasets(        data_root=data_root,        image_dim=image_dim,        **kwargs    )    tr_loader = DataLoader(tr, batch_size=batch_size, shuffle=True)    va_loader = DataLoader(va, batch_size=batch_size, shuffle=False)    te_loader = DataLoader(te, batch_size=batch_size, shuffle=False)    return tr_loader, va_loader, te_loaderclass fMRIDataset(Dataset):    """Dataset class for fMRI brain activation maps from subject folders."""    def __init__(        self,         data_root: str,        subjects: list,        partition: str = "train",        image_dim: int = 224,        standardizer: Optional[ImageStandardizer] = None,        use_imagenet_norm: bool = True    ) -> None:        """Initialize fMRI dataset.                Args:            data_root: Root directory containing subject folders (e.g., dataset_combined_vertical)            subjects: List of subject IDs to include in this dataset            partition: Partition name (for logging)            image_dim: Target image dimension for resizing            standardizer: ImageStandardizer instance (optional)            use_imagenet_norm: If True, use ImageNet normalization (for pretrained models)        """        super().__init__()                self.data_root = data_root        self.subjects = subjects        self.partition = partition        self.image_dim = image_dim        self.standardizer = standardizer        self.use_imagenet_norm = use_imagenet_norm                # ImageNet normalization parameters (for pretrained models)        self.imagenet_mean = np.array([0.485, 0.456, 0.406])        self.imagenet_std = np.array([0.229, 0.224, 0.225])                # Load all images from specified subjects        self.image_paths = []        self.labels = []                print(f"Loading {partition} data from {len(subjects)} subjects...")        for subject in subjects:            subject_dir = os.path.join(data_root, subject, "combined_vertical")            if not os.path.exists(subject_dir):                print(f"Warning: Subject directory not found: {subject_dir}")                continue                        # Load cond1-12 images            for cond in range(1, 13):                image_name = f"brain_cond{cond}_vertical.png"                image_path = os.path.join(subject_dir, image_name)                                if os.path.exists(image_path):                    self.image_paths.append(image_path)                    # cond1-6 = house (1), cond7-12 = face (0)                    label = 1 if cond <= 6 else 0                    self.labels.append(label)                else:                    print(f"Warning: Image not found: {image_path}")                print(f"Loaded {len(self.image_paths)} images for {partition} set")                # Class names        self.class_names = ["face", "house"]        self.label_to_name = {0: "face", 1: "house"}    def __len__(self) -> int:        """Return size of dataset."""        return len(self.image_paths)    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:        """Return (image, label) pair at index `idx` of dataset."""        # Load image        image_path = self.image_paths[idx]        image = imread(image_path)                # Handle grayscale images - convert to RGB if needed        if len(image.shape) == 2:            image = np.stack([image, image, image], axis=-1)        elif len(image.shape) == 3 and image.shape[2] == 1:            image = np.repeat(image, 3, axis=2)        elif len(image.shape) == 3 and image.shape[2] == 4:            # RGBA to RGB            image = image[:, :, :3]                # Resize if needed        if self.image_dim and self.image_dim > 0:            image = resize_image(image, self.image_dim)                # Convert to tensor and normalize to [0, 1]        image = image.astype(np.float32) / 255.0                # Convert to (C, H, W) format        image = image.transpose(2, 0, 1)                # Apply normalization        if self.use_imagenet_norm:            # Use ImageNet normalization for pretrained models            image = (image - self.imagenet_mean[:, None, None]) / self.imagenet_std[:, None, None]        elif self.standardizer is not None:            # Use dataset-specific standardization            # Reshape to (H, W, C) for standardizer            image_hwc = image.transpose(1, 2, 0)            image_hwc = self.standardizer.transform(image_hwc[None, ...])[0]            # Convert back to (C, H, W)            image = image_hwc.transpose(2, 0, 1)                label = self.labels[idx]                return torch.from_numpy(image).float(), torch.tensor(label).long()        def get_class_name(self, label: int) -> str:        """Return class name for a label."""        return self.label_to_name.get(label, f"unknown_{label}")def resize_image(image: np.ndarray, image_dim: int) -> np.ndarray:    """Resize image to specified dimension.        Args:        image: Input image as numpy array        image_dim: Target image dimension (square)            Returns:        Resized image as numpy array    """    image_size = (image_dim, image_dim)    img_pil = Image.fromarray(image.astype(np.uint8))    img_resized = img_pil.resize(image_size, resample=Image.Resampling.BICUBIC)    return np.asarray(img_resized)def get_train_val_test_datasets(    data_root: str,    test_size: float = 0.15,    val_size: float = 0.15,    random_state: int = 445,    image_dim: int = 224) -> tuple[fMRIDataset, fMRIDataset, fMRIDataset, ImageStandardizer]:    """Return fMRIDatasets and image standardizer.        Performs subject-level split (not image-level) to avoid data leakage.    Each subject's images stay in the same partition.        Args:        data_root: Root directory containing subject folders        test_size: Proportion of subjects for test set        val_size: Proportion of subjects for validation set        random_state: Random seed for splitting        image_dim: Target image dimension            Returns:        Tuple of (train_dataset, val_dataset, test_dataset, standardizer)    """    set_random_seed()        # Get all subject folders    all_subjects = [d for d in os.listdir(data_root)                    if os.path.isdir(os.path.join(data_root, d))                   and os.path.exists(os.path.join(data_root, d, "combined_vertical"))]    all_subjects.sort()        print(f"Found {len(all_subjects)} subjects in {data_root}")        # Split subjects into train/val/test    # First split: train+val vs test    train_val_subjects, test_subjects = train_test_split(        all_subjects,        test_size=test_size,        random_state=random_state    )        # Second split: train vs val    val_size_adjusted = val_size / (1 - test_size)    train_subjects, val_subjects = train_test_split(        train_val_subjects,        test_size=val_size_adjusted,        random_state=random_state    )        print(f"Train subjects: {len(train_subjects)}")    print(f"Val subjects: {len(val_subjects)}")    print(f"Test subjects: {len(test_subjects)}")        # Create standardizer (for potential use, but we'll use ImageNet norm by default)    # Create standardizer and fit on training data    print("Computing image statistics from training set...")    # Sample some images to compute statistics    sample_dataset = fMRIDataset(data_root, train_subjects[:10], partition="train", image_dim=image_dim, use_imagenet_norm=False)    train_images = []    for i in range(min(100, len(sample_dataset))):        img, _ = sample_dataset[i]        train_images.append(img.numpy())    train_images = np.array(train_images)        # Reshape to (N, H, W, C) for standardizer    train_images = train_images.transpose(0, 2, 3, 1)        standardizer = ImageStandardizer()    standardizer.fit(train_images)        # Create datasets with ImageNet normalization (for pretrained models)    # Set use_imagenet_norm=True to use ImageNet stats, False to use dataset stats    tr = fMRIDataset(data_root, train_subjects, partition="train", image_dim=image_dim,                      standardizer=standardizer, use_imagenet_norm=True)    va = fMRIDataset(data_root, val_subjects, partition="val", image_dim=image_dim,                     standardizer=standardizer, use_imagenet_norm=True)    te = fMRIDataset(data_root, test_subjects, partition="test", image_dim=image_dim,                     standardizer=standardizer, use_imagenet_norm=True)        return tr, va, te, standardizerif __name__ == "__main__":    # Test dataset loading    data_root = "../dataset_combined_vertical"    tr, va, te, standardizer = get_train_val_test_datasets(data_root)    print(f"\nTrain: {len(tr)} images")    print(f"Val: {len(va)} images")    print(f"Test: {len(te)} images")    print(f"Mean: {standardizer.image_mean}")    print(f"Std: {standardizer.image_std}")

## Model Code

In [ ]:
"""EECS 442 Final Project - BrainMapNetfMRI CNN Classifier    Uses pretrained ResNet-18 or MobileNet for classifying fMRI activation maps"""import torchimport torch.nn as nnimport torchvision.models as modelsfrom torchvision.models import ResNet18_Weights, MobileNet_V3_Small_Weights__all__ = ["fMRICNN"]class fMRICNN(nn.Module):    """CNN classifier using pretrained ResNet-18 or MobileNet for fMRI classification."""        def __init__(        self,         architecture: str = "resnet18",        num_classes: int = 2,        pretrained: bool = True,        freeze_backbone: bool = False    ) -> None:        """Initialize the fMRI CNN classifier.                Args:            architecture: Either "resnet18" or "mobilenet"            num_classes: Number of output classes (default: 2 for face/house)            pretrained: Whether to use pretrained weights            freeze_backbone: Whether to freeze the backbone (for fine-tuning)        """        super().__init__()                self.architecture = architecture.lower()        self.num_classes = num_classes                if self.architecture == "resnet18":            if pretrained:                self.backbone = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)            else:                self.backbone = models.resnet18(weights=None)                        # Replace the final fully connected layer            num_features = self.backbone.fc.in_features            self.backbone.fc = nn.Linear(num_features, num_classes)                    elif self.architecture == "mobilenet":            if pretrained:                self.backbone = models.mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)            else:                self.backbone = models.mobilenet_v3_small(weights=None)                        # Replace the classifier            num_features = self.backbone.classifier[-1].in_features            self.backbone.classifier[-1] = nn.Linear(num_features, num_classes)                    else:            raise ValueError(f"Unsupported architecture: {architecture}. Choose 'resnet18' or 'mobilenet'")                # Freeze backbone if specified        if freeze_backbone:            for param in self.backbone.parameters():                param.requires_grad = False            # Unfreeze the final layer            if self.architecture == "resnet18":                for param in self.backbone.fc.parameters():                    param.requires_grad = True            else:  # mobilenet                for param in self.backbone.classifier[-1].parameters():                    param.requires_grad = True        def forward(self, x: torch.Tensor) -> torch.Tensor:        """Forward pass."""        return self.backbone(x)        def get_features(self, x: torch.Tensor) -> torch.Tensor:        """Extract features before the final classification layer.                Useful for Grad-CAM visualization.        """        if self.architecture == "resnet18":            x = self.backbone.conv1(x)            x = self.backbone.bn1(x)            x = self.backbone.relu(x)            x = self.backbone.maxpool(x)                        x = self.backbone.layer1(x)            x = self.backbone.layer2(x)            x = self.backbone.layer3(x)            x = self.backbone.layer4(x)                        x = self.backbone.avgpool(x)            x = torch.flatten(x, 1)            return x        else:  # mobilenet            x = self.backbone.features(x)            x = self.backbone.avgpool(x)            x = torch.flatten(x, 1)            return x

## Grad-CAM Code

In [ ]:
"""EECS 442 Final Project - BrainMapNetGrad-CAM Visualization    Implementation of Grad-CAM for visualizing important regions in fMRI activation maps"""import torchimport torch.nn.functional as Fimport numpy as npimport matplotlib.pyplot as pltimport cv2__all__ = ["GradCAM", "visualize_gradcam"]class GradCAM:    """Grad-CAM implementation for CNN models."""        def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):        """Initialize Grad-CAM.                Args:            model: The trained model            target_layer: The target convolutional layer to visualize        """        self.model = model        self.target_layer = target_layer        self.gradients = None        self.activations = None                # Register hooks        self.target_layer.register_forward_hook(self._save_activation)        self.target_layer.register_full_backward_hook(self._save_gradient)        def _save_activation(self, module, input, output):        """Save activation maps."""        self.activations = output.detach()        def _save_gradient(self, module, grad_input, grad_output):        """Save gradients."""        self.gradients = grad_output[0].detach()        def generate_cam(self, input_tensor: torch.Tensor, class_idx: int = None) -> np.ndarray:        """Generate Class Activation Map (CAM).                Args:            input_tensor: Input image tensor of shape (1, C, H, W)            class_idx: Class index to visualize. If None, uses predicted class.                    Returns:            CAM heatmap as numpy array        """        self.model.eval()                # Forward pass        output = self.model(input_tensor)                if class_idx is None:            class_idx = output.argmax(dim=1).item()                # Backward pass        self.model.zero_grad()        output[0, class_idx].backward()                # Get gradients and activations        gradients = self.gradients[0]  # Shape: (C, H, W)        activations = self.activations[0]  # Shape: (C, H, W)                # Calculate weights (global average pooling of gradients)        weights = torch.mean(gradients, dim=(1, 2))  # Shape: (C,)                # Weighted combination of activation maps        cam = torch.zeros(activations.shape[1:], dtype=torch.float32)        for i, w in enumerate(weights):            cam += w * activations[i, :, :]                # Apply ReLU        cam = F.relu(cam)                # Normalize        cam = cam - cam.min()        cam = cam / (cam.max() + 1e-8)                return cam.cpu().numpy()def visualize_gradcam(    model: torch.nn.Module,    input_tensor: torch.Tensor,    original_image: np.ndarray,    class_names: list = None,    save_path: str = None,    architecture: str = "resnet18") -> None:    """Visualize Grad-CAM heatmap overlaid on original image.        Args:        model: Trained model        input_tensor: Input tensor of shape (1, C, H, W)        original_image: Original image as numpy array (H, W, C)        class_names: List of class names (e.g., ["face", "house"])        save_path: Path to save the visualization        architecture: Model architecture ("resnet18" or "mobilenet")    """    model.eval()        # Get prediction    with torch.no_grad():        output = model(input_tensor)        pred_class = output.argmax(dim=1).item()        pred_prob = F.softmax(output, dim=1)[0, pred_class].item()        # Initialize Grad-CAM    if architecture == "resnet18":        if hasattr(model, 'backbone'):            target_layer = model.backbone.layer4        else:            target_layer = model.layer4    else:  # mobilenet        if hasattr(model, 'backbone'):            features = model.backbone.features            target_layer = features[-1]        else:            target_layer = model.features[-1]        gradcam = GradCAM(model, target_layer)        # Generate CAM    cam = gradcam.generate_cam(input_tensor, pred_class)        # Resize CAM to match original image size    cam_resized = cv2.resize(cam, (original_image.shape[1], original_image.shape[0]))    cam_resized = np.uint8(255 * cam_resized)    heatmap = cv2.applyColorMap(cam_resized, cv2.COLORMAP_JET)        # Convert original image to RGB if needed    if len(original_image.shape) == 2:        original_image = np.stack([original_image, original_image, original_image], axis=-1)    elif original_image.shape[2] == 1:        original_image = np.repeat(original_image, 3, axis=2)        # Normalize original image to [0, 1]    if original_image.max() > 1:        original_image = original_image.astype(np.float32) / 255.0        # Overlay heatmap on original image    heatmap = heatmap.astype(np.float32) / 255.0    overlayed = 0.6 * original_image + 0.4 * heatmap        # Create visualization    fig, axes = plt.subplots(1, 3, figsize=(15, 5))        # Original image    axes[0].imshow(original_image)    axes[0].set_title("Original fMRI Map")    axes[0].axis('off')        # Heatmap    axes[1].imshow(cam_resized, cmap='jet')    axes[1].set_title("Grad-CAM Heatmap")    axes[1].axis('off')        # Overlay    axes[2].imshow(overlayed)    class_name = class_names[pred_class] if class_names else f"Class {pred_class}"    axes[2].set_title(f"Overlay\nPredicted: {class_name} ({pred_prob:.3f})")    axes[2].axis('off')        plt.tight_layout()        if save_path:        plt.savefig(save_path, dpi=200, bbox_inches='tight')        print(f"Grad-CAM visualization saved to {save_path}")        plt.close()

## Training Utilities

In [ ]:
"""EECS 445 - Introduction to Machine LearningWinter 2025 - Project 2Helper file for common training functions."""import itertoolsimport osimport matplotlibimport numpy as npimport torchfrom torch import nnfrom torch.nn.functional import softmaxfrom sklearn import metricsimport utils__all__ = [    "count_parameters",    "save_checkpoint",    "restore_checkpoint",    "clear_checkpoint",    "early_stopping",    "evaluate_epoch",    "train_epoch",    "predictions",]def count_parameters(model: torch.nn.Module) -> int:    """Count number of learnable parameters."""    return sum(p.numel() for p in model.parameters() if p.requires_grad)def save_checkpoint(model: torch.nn.Module, epoch: int, checkpoint_dir: str, stats: list) -> None:    """Save a checkpoint file to `checkpoint_dir`."""    state = {        "epoch": epoch,        "state_dict": model.state_dict(),        "stats": stats,    }    filename = os.path.join(checkpoint_dir, "epoch={}.checkpoint.pth.tar".format(epoch))    if not os.path.exists(checkpoint_dir):        os.makedirs(checkpoint_dir,exist_ok=True)    torch.save(state, filename)def restore_checkpoint(    model: torch.nn.Module,     checkpoint_dir: str,     cuda: bool=False,     force: bool=False,     pretrain: bool=False) -> tuple[nn.Module, int, list]:    """    Restore model from checkpoint if it exists.    Args:        model: The model to be restored.        checkpoint_dir: Directory where checkpoint files are stored.        cuda: Whether to load the model on GPU if available. Defaults to False.        force: If True, force the user to choose an epoch. Defaults to False.        pretrain: If True, allows partial loading of the model state (used for pretraining). Defaults to False.    Returns:        tuple: The restored model, the starting epoch, and the list of statistics.    Description:        This function attempts to restore a saved model from the specified `checkpoint_dir`.        If no checkpoint is found, the function either raises an exception (if `force` is True) or returns        the original model and starts from epoch 0. If a checkpoint is available, the user can choose which        epoch to load from. The model's parameters, epoch number, and training statistics are restored.    """    try:        cp_files = [            file_            for file_ in os.listdir(checkpoint_dir)            if file_.startswith("epoch=") and file_.endswith(".checkpoint.pth.tar")        ]    except FileNotFoundError:        cp_files = None        os.makedirs(checkpoint_dir)    if not cp_files:        print("No saved model parameters found")        if force:            raise Exception("Checkpoint not found")        else:            return model, 0, []    # Find latest epoch    for i in itertools.count(1):        if "epoch={}.checkpoint.pth.tar".format(i) in cp_files:            epoch = i        else:            break    if not force:        print(            "Which epoch to load from? Choose in range [0, {}].".format(epoch),            "Enter 0 to train from scratch.",        )        print(">> ", end="")        inp_epoch = int(input())        if inp_epoch not in range(epoch + 1):            raise Exception("Invalid epoch number")        if inp_epoch == 0:            print("Checkpoint not loaded")            clear_checkpoint(checkpoint_dir)            return model, 0, []    else:        print("Which epoch to load from? Choose in range [1, {}].".format(epoch))        inp_epoch = int(input())        if inp_epoch not in range(1, epoch + 1):            raise Exception("Invalid epoch number")    filename = os.path.join(        checkpoint_dir, "epoch={}.checkpoint.pth.tar".format(inp_epoch)    )    print("Loading from checkpoint {}?".format(filename))    if cuda:        checkpoint = torch.load(filename, weights_only=False)    else:        # Load GPU model on CPU        checkpoint = torch.load(filename, map_location=lambda storage, loc: storage, weights_only=False)    try:        start_epoch = checkpoint["epoch"]        stats = checkpoint["stats"]        if pretrain:            model.load_state_dict(checkpoint["state_dict"], strict=False)        else:            model.load_state_dict(checkpoint["state_dict"])        print(            "=> Successfully restored checkpoint (trained for {} epochs)".format(                checkpoint["epoch"]            )        )    except:        print("=> Checkpoint not successfully restored")        raise    return model, inp_epoch, statsdef clear_checkpoint(checkpoint_dir: str) -> None:    """Remove checkpoints in `checkpoint_dir`."""    filelist = [f for f in os.listdir(checkpoint_dir) if f.endswith(".pth.tar")]    for f in filelist:        os.remove(os.path.join(checkpoint_dir, f))    print("Checkpoint successfully removed")def early_stopping(stats: list, curr_count_to_patience: int, prev_val_loss: float) -> tuple[int, float]:    """Calculate new patience and validation loss.    Increment curr_count_to_patience by one if new loss is not less than prev_val_loss    Otherwise, update prev_val_loss with the current val loss, and reset curr_count_to_patience to 0    Returns: new values of curr_count_to_patience and prev_val_loss    """    # TODO: 2(e) - implement early stopping    ###### PART 2(e): IMPLEMENTED early_stopping() function ######    # Get the latest validation loss (stats[-1][1] is validation loss)    curr_val_loss = stats[-1][1]        if curr_val_loss < prev_val_loss:        # Validation loss improved, reset patience and update best loss        curr_count_to_patience = 0        prev_val_loss = curr_val_loss    else:        # Validation loss didn't improve, increment patience        curr_count_to_patience += 1        return curr_count_to_patience, prev_val_lossdef evaluate_epoch(    axes: matplotlib.axes.Axes,    tr_loader: torch.utils.data.DataLoader,    val_loader: torch.utils.data.DataLoader,    te_loader: torch.utils.data.DataLoader,    model: torch.nn.Module,    criterion: torch.nn.Module,    epoch: int,    stats: list,    include_test: bool = False,    update_plot: bool = True,    multiclass: bool = False,) -> None:    """Evaluate the `model` on the train and validation set."""    def _get_metrics(loader):        y_true, y_pred, y_score = [], [], []        correct, total = 0, 0        running_loss = []        for X, y in loader:            with torch.no_grad():                output = model(X)                predicted = predictions(output.data)                y_true.append(y)                y_pred.append(predicted)                if not multiclass:                    y_score.append(softmax(output.data, dim=1)[:, 1])                else:                    y_score.append(softmax(output.data, dim=1))                total += y.size(0)                correct += (predicted == y).sum().item()                running_loss.append(criterion(output, y).item())        y_true = torch.cat(y_true)        y_pred = torch.cat(y_pred)        y_score = torch.cat(y_score)        loss = np.mean(running_loss)        acc = correct / total        if not multiclass:            auroc = metrics.roc_auc_score(y_true, y_score)        else:            auroc = metrics.roc_auc_score(y_true, y_score, multi_class="ovo")        return acc, loss, auroc    train_acc, train_loss, train_auc = _get_metrics(tr_loader)    val_acc, val_loss, val_auc = _get_metrics(val_loader)    stats_at_epoch = [        val_acc,        val_loss,        val_auc,        train_acc,        train_loss,        train_auc,    ]    if include_test:        stats_at_epoch += list(_get_metrics(te_loader))    stats.append(stats_at_epoch)    utils.log_training(epoch, stats)    if update_plot:        utils.update_training_plot(axes, epoch, stats)def train_epoch(    data_loader: torch.utils.data.DataLoader,    model: torch.nn.Module,    criterion: torch.nn.Module,    optimizer: torch.optim.Optimizer,) -> None:    """Train the `model` for one epoch of data from `data_loader`.    Args:        data_loader: DataLoader providing batches of input data and corresponding labels.        model: The model to be trained. This is one of the model classes in the 'model' folder.         criterion: The loss function used to compute the model's loss.        optimizer: The optimizer used to update the model parameters.    Description:        This function sets the model to training mode and use the data loader to iterate through the entire dataset.        For each batch, it performs the following steps:        1. Resets the gradient calculations in the optimizer.        2. Performs a forward pass to get the model predictions.        3. Computes the loss between predictions and true labels using the specified `criterion`.        4. Performs a backward pass to calculate gradients.        5. Updates the model weights using the `optimizer`.    """    model.train()  # Set model to training mode    for i, (X, y) in enumerate(data_loader):        # TODO 2(e) - implement training steps:        # 1. Reset optimizer gradient calculations        optimizer.zero_grad()                # 2. Get model predictions        output = model(X)                # 3. Calculate loss between model prediction and true labels        loss = criterion(output, y)                # 4. Perform backward pass        loss.backward()                # 5. Update model weights        optimizer.step()def predictions(logits: torch.Tensor) -> torch.Tensor:    """Determine predicted class index given logits.    args:         logits: The model's output logits. It is a 2D tensor of shape (batch_size, num_classes).     Returns:        the predicted class output that has the highest probability. This should be of size (batch_size,).    """    # TODO 2(c) - implement predictions    return torch.argmax(logits, dim=1)

## Utilities

In [ ]:
"""EECS 445 - Introduction to Machine LearningWinter 2025 - Project 2Utility functions"""import randomfrom typing import Anyimport torchimport numpy as npimport matplotlib.pyplot as plt__all__ = [    "set_random_seed",    "config",    "denormalize_image",    "log_training",    "make_training_plot",    "update_training_plot",]# DO NOT CHANGE THIS VARIABLE!SEED = 445def set_random_seed() -> None:    """Set the random seed for reproducibility and enforces deterministic algorithms.        DO NOT MODIFY THIS FUNCTION!    """    random.seed(SEED)    np.random.seed(SEED)    torch.manual_seed(SEED)    torch.cuda.manual_seed(SEED)    torch.use_deterministic_algorithms(True)    torch.backends.cudnn.deterministic = True    torch.backends.cudnn.benchmark = Falsedef config(attr: str) -> Any:    """    Retrieves the queried attribute value from the config file. Loads the    config file on first call.    Args:        attr: the attribute to retrieve from the config file    Returns:        the value of the attribute in the config file    """    if not hasattr(config, "config"):        with open("config.json") as f:            config.config = eval(f.read())    node = config.config    for part in attr.split("."):        node = node[part]    return nodedef denormalize_image(image: np.ndarray) -> np.ndarray:    """    Rescale the image's color space from (min, max) to (0, 1)        Args:        image: the image to denormalize            Returns:        the denormalized image    """    ptp = np.max(image, axis=(0, 1)) - np.min(image, axis=(0, 1))    return (image - np.min(image, axis=(0, 1))) / ptpdef log_training(epoch: int, stats: list[list[float]]) -> None:    """Print the train, validation, test accuracy/loss/auroc.    Args:        stats: A cumulative list to store the model accuracy, loss, and AUC for every epoch.            Usage: stats[epoch][0] = validation accuracy, stats[epoch][1] = validation loss, stats[epoch][2] = validation AUC                    stats[epoch][3] = training accuracy, stats[epoch][4] = training loss, stats[epoch][5] = training AUC                    stats[epoch][6] = test accuracy, stats[epoch][7] = test loss, stats[epoch][8] = test AUC (test only appears when we are finetuning our target model)            epoch: The current epoch number.        Note: Test accuracy is optional and will only be logged if stats is length 9.    """    splits = ["Validation", "Train", "Test"]    metrics = ["Accuracy", "Loss", "AUROC"]    print("Epoch {}".format(epoch))    for j, split in enumerate(splits):        for i, metric in enumerate(metrics):            idx = len(metrics) * j + i            if idx >= len(stats[-1]):                continue            print(f"\t{split} {metric}:{round(stats[-1][idx],4)}")def make_training_plot(name: str = "CNN Training") -> plt.Axes:    """    Set up an interactive matplotlib graph to log metrics during training.        Args:        name: The name of the training plot.        Returns:        axes: The axes of the training    """    plt.ion()    fig, axes = plt.subplots(1, 3, figsize=(17, 5))    plt.suptitle(name)    axes[0].set_xlabel("Epoch")    axes[0].set_ylabel("Accuracy")    axes[1].set_xlabel("Epoch")    axes[1].set_ylabel("Loss")    axes[2].set_xlabel("Epoch")    axes[2].set_ylabel("AUROC")    return axesdef update_training_plot(axes: plt.Axes, epoch: int, stats: list[list[float]]) -> None:    """Update the training plot with a new data point for loss and accuracy."""    splits = ["Validation", "Train", "Test"]    metrics = ["Accuracy", "Loss", "AUROC"]    colors = ["r", "b", "y"]    styles = ["o", "x", "^"]    for i, metric in enumerate(metrics):        for j, split in enumerate(splits):            idx = len(metrics) * j + i            if idx >= len(stats[-1]):                continue            axes[i].plot(                range(epoch - len(stats) + 1, epoch + 1),                [stat[idx] for stat in stats],                linestyle="--",                marker=styles[j],                color=colors[j],            )        axes[i].legend(splits[: int(len(stats[-1]) / len(metrics))])    plt.pause(0.00001)

## Evaluation Code

In [ ]:
"""EECS 442 Final Project - BrainMapNetEvaluate fMRI Models    Evaluate trained models and generate confusion matrix and Grad-CAM visualizations    Usage: python evaluate.py --model cnn --architecture resnet18"""import argparseimport torchimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.metrics import confusion_matrix, classification_report, f1_score, roc_auc_scorefrom torch.nn.functional import softmaximport seaborn as snsimport sysimport os# Add parent directory to path for importssys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))from dataset import get_train_val_test_loaders, get_train_val_test_datasetsfrom model.fmri_cnn import fMRICNNfrom model.fmri_mlp import fMRIMLPfrom train_common import restore_checkpoint, predictionsfrom utils import set_random_seedfrom gradcam import visualize_gradcamdef evaluate_model(model, loader, criterion, device='cpu'):    """Evaluate model on a dataset."""    model.eval()    y_true, y_pred, y_score = [], [], []    total_loss = 0.0        with torch.no_grad():        for X, y in loader:            X, y = X.to(device), y.to(device)            output = model(X)            loss = criterion(output, y)                        total_loss += loss.item()            y_true.append(y.cpu())            y_pred.append(predictions(output).cpu())            y_score.append(softmax(output, dim=1)[:, 1].cpu())        y_true = torch.cat(y_true).numpy()    y_pred = torch.cat(y_pred).numpy()    y_score = torch.cat(y_score).numpy()        accuracy = (y_true == y_pred).mean()    avg_loss = total_loss / len(loader)    f1 = f1_score(y_true, y_pred)    auroc = roc_auc_score(y_true, y_score)        return {        'accuracy': accuracy,        'loss': avg_loss,        'f1': f1,        'auroc': auroc,        'y_true': y_true,        'y_pred': y_pred,        'y_score': y_score    }def plot_confusion_matrix(y_true, y_pred, class_names, save_path=None):    """Plot confusion matrix."""    cm = confusion_matrix(y_true, y_pred)        plt.figure(figsize=(8, 6))    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',                 xticklabels=class_names, yticklabels=class_names)    plt.ylabel('True Label')    plt.xlabel('Predicted Label')    plt.title('Confusion Matrix')    plt.tight_layout()        if save_path:        plt.savefig(save_path, dpi=200)        print(f"Confusion matrix saved to {save_path}")        plt.show()def visualize_samples_with_gradcam(model, loader, class_names, num_samples=5,                                    architecture='resnet18', save_dir='gradcam_visualizations'):    """Visualize sample predictions with Grad-CAM."""    os.makedirs(save_dir, exist_ok=True)        model.eval()    samples_visualized = 0        for X, y in loader:        if samples_visualized >= num_samples:            break                for i in range(X.size(0)):            if samples_visualized >= num_samples:                break                        # Get single sample            x_sample = X[i:i+1]            y_true = y[i].item()                        # Get original image (denormalize if needed)            img = x_sample[0].permute(1, 2, 0).cpu().numpy()            # Denormalize if needed            if img.min() < 0:                img = (img - img.min()) / (img.max() - img.min() + 1e-8)            elif img.max() > 1:                img = img / 255.0                        # Visualize            save_path = os.path.join(save_dir, f"sample_{samples_visualized}_true_{class_names[y_true]}.png")            visualize_gradcam(                model, x_sample, img,                 class_names=class_names,                save_path=save_path,                architecture=architecture            )                        samples_visualized += 1def main():    parser = argparse.ArgumentParser(description='Evaluate fMRI models')    parser.add_argument('--model', type=str, choices=['cnn', 'mlp'], default='cnn',                        help='Model type to evaluate')    parser.add_argument('--architecture', type=str, choices=['resnet18', 'mobilenet'],                         default='resnet18', help='CNN architecture (only for CNN model)')    parser.add_argument('--data_root', type=str, default=DATA_ROOT,                        help='Root directory containing subject folders')    parser.add_argument('--device', type=str, default='cpu', help='Device to use')    parser.add_argument('--visualize', action='store_true',                         help='Generate Grad-CAM visualizations')    parser.add_argument('--num_samples', type=int, default=5,                        help='Number of samples to visualize with Grad-CAM')    parser.add_argument('--checkpoint_dir', type=str, default=None,                        help='Checkpoint directory (default: ./checkpoints/fmri_{model}/)')        args = parser.parse_args()        set_random_seed()    device = torch.device(args.device)        # Get class names    class_names = ["face", "house"]        # Load model    if args.model == 'cnn':        model = fMRICNN(            architecture=args.architecture,            num_classes=2,            pretrained=True,            freeze_backbone=False        )        checkpoint_dir = args.checkpoint_dir or "./checkpoints/fmri_cnn/"                # Load data        tr_loader, va_loader, te_loader = get_train_val_test_loaders(            data_root=args.data_root,            batch_size=16,            image_dim=224        )    else:  # mlp        model = fMRIMLP(            input_dim=2000,            hidden_dims=[512, 256],            num_classes=2,            dropout_rate=0.3        )        checkpoint_dir = args.checkpoint_dir or "./checkpoints/fmri_mlp/"                # For MLP, we need to use the voxel extraction        from train_mlp import get_mlp_loaders        tr_loader, va_loader, te_loader = get_mlp_loaders(            data_root=args.data_root,            batch_size=32,            top_k=2000,            image_dim=224        )        # Restore checkpoint    print(f"Loading checkpoint from {checkpoint_dir}...")    model, epoch, stats = restore_checkpoint(model, checkpoint_dir, force=True)    model = model.to(device)        print(f"Loaded model from epoch {epoch}")        # Loss function    criterion = torch.nn.CrossEntropyLoss()        # Evaluate on all splits    print("\n" + "="*50)    print("EVALUATION RESULTS")    print("="*50)        for split_name, loader in [("Train", tr_loader), ("Validation", va_loader), ("Test", te_loader)]:        results = evaluate_model(model, loader, criterion, device)        print(f"\n{split_name} Set:")        print(f"  Accuracy: {results['accuracy']:.4f}")        print(f"  Loss: {results['loss']:.4f}")        print(f"  F1 Score: {results['f1']:.4f}")        print(f"  ROC-AUC: {results['auroc']:.4f}")        # Confusion matrix on test set    test_results = evaluate_model(model, te_loader, criterion, device)    plot_confusion_matrix(        test_results['y_true'],         test_results['y_pred'],        class_names,        save_path=f"fmri_{args.model}_confusion_matrix.png"    )        # Classification report    print("\n" + "="*50)    print("CLASSIFICATION REPORT (Test Set)")    print("="*50)    print(classification_report(        test_results['y_true'],        test_results['y_pred'],        target_names=class_names    ))        # Grad-CAM visualizations (only for CNN)    if args.visualize and args.model == 'cnn':        print("\nGenerating Grad-CAM visualizations...")        visualize_samples_with_gradcam(            model, te_loader, class_names,            num_samples=args.num_samples,            architecture=args.architecture        )if __name__ == "__main__":    main()

## Run Evaluation

In [ ]:
# Load model and evaluate# Modify the code above to match your setup# Then run the evaluation functions